## Step 1

In [1]:
import pandas as pd
import networkx as nx

df = pd.read_csv("data/raw/twitter_edges.csv")
G = nx.from_pandas_edgelist(df, "src", "dst")

# 先取前 3000 个节点做实验
nodes = list(G.nodes())[:3000]
G_small = G.subgraph(nodes).copy()

df_small = nx.to_pandas_edgelist(G_small)
df_small.columns = ["src", "dst"]
df_small.to_csv("data/raw/twitter_edges_small.csv", index=False)

print("已生成 twitter_edges_small.csv")
print("Nodes:", G_small.number_of_nodes())
print("Edges:", G_small.number_of_edges())

已生成 twitter_edges_small.csv
Nodes: 3000
Edges: 50199


## Step 2

In [2]:
import os
import sys
sys.path.append(os.path.abspath("src"))

from data_utils import *
from noise_utils import *
from matching import *
from metrics import *
from baselines import *
from models import *
from experiment import *

C:\Users\23293\anaconda3\envs\netalign\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3

In [3]:
set_seed(42)

G = load_base_graph(
    name="twitter",
    data_dir="data/raw",
    edge_file="twitter_edges_small.csv"
)

print("Base graph loaded.")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Base graph loaded.
Nodes: 2993
Edges: 50187


## Step 4

In [5]:
X = generate_node_features(G, feat_dim=16, mode="structural")
print("Feature shape:", X.shape)

Feature shape: (2993, 16)


## Step 5

In [6]:
Gs, Gt, Xs, Xt, gt_map = generate_permuted_graph_pair(G, X)

print("Source nodes:", Gs.number_of_nodes(), "edges:", Gs.number_of_edges())
print("Target nodes:", Gt.number_of_nodes(), "edges:", Gt.number_of_edges())
print("GT pairs:", len(gt_map))

Source nodes: 2993 edges: 50187
Target nodes: 2993 edges: 50187
GT pairs: 2993


## Step 6

In [7]:
As, src_nodes = graph_to_adjacency(Gs)
At, tgt_nodes = graph_to_adjacency(Gt)
gt_idx = build_gt_index_map(src_nodes, tgt_nodes, gt_map)

print("Adjacency shapes:", As.shape, At.shape)
print("GT index shape:", gt_idx.shape)

Adjacency shapes: (2993, 2993) (2993, 2993)
GT index shape: (2993,)


## No Noise Sanity Check_1. Node2Vec baseline

In [8]:
(Zs, _), t1 = timed_run(node2vec_embed, Gs, 64)
(Zt, _), t2 = timed_run(node2vec_embed, Gt, 64)

sim = similarity_matrix(Zs, Zt)
pred_nn = match_nearest_neighbor(sim)

print("Node2Vec Accuracy@1:", accuracy_at_1(pred_nn, gt_idx))
print("Node2Vec Hit@1:", hit_at_k(sim, gt_idx, 1))
print("Node2Vec Hit@5:", hit_at_k(sim, gt_idx, 5))
print("Runtime:", t1 + t2)

Generating walks (CPU: 1): 100%|██████████| 50/50 [00:16<00:00,  3.07it/s]


Node2Vec Accuracy@1: 0.0006682258603407952
Node2Vec Hit@1: 0.0006682258603407952
Node2Vec Hit@5: 0.0020046775810223854
Runtime: 135.16358923912048


## _2. FINAL-style baseline

In [10]:
final = FINALAligner(alpha=0.8, max_iter=30).fit(As, At, Xs, Xt)
S = final.predict_score_matrix()
pred_final = final.predict_top1(use_hungarian=False)

print("FINAL Accuracy@1:", accuracy_at_1(pred_final, gt_idx))
print("FINAL Hit@1:", hit_at_k(S, gt_idx, 1))
print("FINAL Hit@5:", hit_at_k(S, gt_idx, 5))

FINAL Accuracy@1: 0.0006682258603407952
FINAL Hit@1: 0.0006682258603407952
FINAL Hit@5: 0.0023387905111927833


## _3. WAlign-inspired / LightGCN baseline

In [11]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

As_t = torch.tensor(As, dtype=torch.float32, device=device)
At_t = torch.tensor(At, dtype=torch.float32, device=device)
Xs_t = torch.tensor(Xs, dtype=torch.float32, device=device)
Xt_t = torch.tensor(Xt, dtype=torch.float32, device=device)

model = LightGCNAlign(
    in_dim=Xs.shape[1],
    hidden_dim=64,
    out_dim=64,
    num_hops=2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(20):
    stats = train_walign_epoch(model, As_t, Xs_t, At_t, Xt_t, optimizer)
    if epoch % 5 == 0:
        print(epoch, stats)

model.eval()
with torch.no_grad():
    Zs_t, Zt, _, _ = model(As_t, Xs_t, At_t, Xt_t)
    sim_t = torch.matmul(Zs_t, Zt.T).cpu().numpy()

pred_model = sim_t.argmax(axis=1)

print("WAlign-style Accuracy@1:", accuracy_at_1(pred_model, gt_idx))
print("WAlign-style Hit@1:", hit_at_k(sim_t, gt_idx, 1))
print("WAlign-style Hit@5:", hit_at_k(sim_t, gt_idx, 5))

0 {'loss': 1.905792236328125, 'align_loss': 0.01419944316148758, 'rec_loss': 1.8915927410125732}
5 {'loss': 1.876387357711792, 'align_loss': 0.011626002378761768, 'rec_loss': 1.8647613525390625}
10 {'loss': 1.8544145822525024, 'align_loss': 0.00969718862324953, 'rec_loss': 1.8447173833847046}
15 {'loss': 1.83779776096344, 'align_loss': 0.007658894639462233, 'rec_loss': 1.830138921737671}
WAlign-style Accuracy@1: 0.0023387905111927833
WAlign-style Hit@1: 0.0023387905111927833
WAlign-style Hit@5: 0.011025726695623121


## Step 7

In [12]:
Gs_n, Gt_n, Xs_n, Xt_n = build_noisy_pair(
    Gs, Gt, Xs, Xt,
    edge_noise=0.1,
    attr_noise=0.1
)

As_n, src_nodes_n = graph_to_adjacency(Gs_n)
At_n, tgt_nodes_n = graph_to_adjacency(Gt_n)
gt_idx_n = build_gt_index_map(src_nodes_n, tgt_nodes_n, gt_map)

## 7_1. noisy Node2Vec

In [13]:
# noisy Node2Vec
(Zs_n, _), t1_n = timed_run(node2vec_embed, Gs_n, 64)
(Zt_n, _), t2_n = timed_run(node2vec_embed, Gt_n, 64)

sim_n = similarity_matrix(Zs_n, Zt_n)
pred_nn_n = match_nearest_neighbor(sim_n)

print("Noisy Node2Vec Accuracy@1:", accuracy_at_1(pred_nn_n, gt_idx_n))
print("Noisy Node2Vec Hit@1:", hit_at_k(sim_n, gt_idx_n, 1))
print("Noisy Node2Vec Hit@5:", hit_at_k(sim_n, gt_idx_n, 5))
print("Noisy Node2Vec Runtime:", t1_n + t2_n)

Generating walks (CPU: 1): 100%|██████████| 50/50 [00:16<00:00,  3.12it/s]


Noisy Node2Vec Accuracy@1: 0.0010023387905111927
Noisy Node2Vec Hit@1: 0.0010023387905111927
Noisy Node2Vec Hit@5: 0.005011693952555964
Noisy Node2Vec Runtime: 125.78828120231628


## 7_2. noisy FINAL

In [14]:
# noisy FINAL
final_n = FINALAligner(alpha=0.8, max_iter=30).fit(As_n, At_n, Xs_n, Xt_n)
S_n = final_n.predict_score_matrix()
pred_final_n = final_n.predict_top1(use_hungarian=False)

print("Noisy FINAL Accuracy@1:", accuracy_at_1(pred_final_n, gt_idx_n))
print("Noisy FINAL Hit@1:", hit_at_k(S_n, gt_idx_n, 1))
print("Noisy FINAL Hit@5:", hit_at_k(S_n, gt_idx_n, 5))

Noisy FINAL Accuracy@1: 0.0013364517206815904
Noisy FINAL Hit@1: 0.0013364517206815904
Noisy FINAL Hit@5: 0.003341129301703976


## 7_3. noisy WAlign-style

In [15]:
# noisy WAlign-style
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

As_n_t = torch.tensor(As_n, dtype=torch.float32, device=device)
At_n_t = torch.tensor(At_n, dtype=torch.float32, device=device)
Xs_n_t = torch.tensor(Xs_n, dtype=torch.float32, device=device)
Xt_n_t = torch.tensor(Xt_n, dtype=torch.float32, device=device)

model_n = LightGCNAlign(
    in_dim=Xs_n.shape[1],
    hidden_dim=64,
    out_dim=64,
    num_hops=2
).to(device)

optimizer_n = torch.optim.Adam(model_n.parameters(), lr=1e-3)

for epoch in range(20):
    stats_n = train_walign_epoch(model_n, As_n_t, Xs_n_t, At_n_t, Xt_n_t, optimizer_n)
    if epoch % 5 == 0:
        print(epoch, stats_n)

model_n.eval()
with torch.no_grad():
    Zs_n_t, Zt_n_t, _, _ = model_n(As_n_t, Xs_n_t, At_n_t, Xt_n_t)
    sim_t_n = torch.matmul(Zs_n_t, Zt_n_t.T).cpu().numpy()

pred_model_n = sim_t_n.argmax(axis=1)

print("Noisy WAlign-style Accuracy@1:", accuracy_at_1(pred_model_n, gt_idx_n))
print("Noisy WAlign-style Hit@1:", hit_at_k(sim_t_n, gt_idx_n, 1))
print("Noisy WAlign-style Hit@5:", hit_at_k(sim_t_n, gt_idx_n, 5))

0 {'loss': 1.7338173389434814, 'align_loss': 0.01449712086468935, 'rec_loss': 1.7193201780319214}
5 {'loss': 1.7046242952346802, 'align_loss': 0.01238317135721445, 'rec_loss': 1.6922410726547241}
10 {'loss': 1.682013750076294, 'align_loss': 0.010504013858735561, 'rec_loss': 1.6715097427368164}
15 {'loss': 1.665177822113037, 'align_loss': 0.008861870504915714, 'rec_loss': 1.6563159227371216}
Noisy WAlign-style Accuracy@1: 0.0006682258603407952
Noisy WAlign-style Hit@1: 0.0006682258603407952
Noisy WAlign-style Hit@5: 0.0013364517206815904
